In [1]:

# Analysis Plan: Three-Class GEV Analysis of R_comp Extreme Value Distributions
# 
# STEP 1: Define function classes and implementation strategy
# - Standard: ζ(s), L(s,χ₄) (real character mod 5)
# - Anomalous-Resonant: L(s, λ(n)), L(s, μ(n)), validated historical L_DH
# - Control: f_rand (seed=42), f_fully_rand (seed=42)
# 
# STEP 2: Generate R_comp(t) time series for each function
# - Use N=10^5 truncation
# - Time range t ∈ [1000, 20000]
# - Compute R_comp using sliding window
# 
# STEP 3: Extract block maxima (200 blocks per function)
# 
# STEP 4: Fit GEV distribution to block maxima
# - Estimate shape parameter ξ with 95% CI via bootstrap
# 
# STEP 5: Statistical comparison
# - Create grouped boxplot of ξ estimates
# - Perform ANOVA + post-hoc tests (Tukey's HSD)
# 
# STEP 6: Interpret results in context of hypothesis

# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.optimize import minimize
from scipy.special import gammaln
import warnings
warnings.filterwarnings('ignore')

print("Analysis plan defined. Beginning implementation...")
print(f"Python version: {np.version.version}")


Analysis plan defined. Beginning implementation...
Python version: 1.26.4


In [2]:

# STEP 1: Define coefficient functions for each L-function class
# We need to compute D_F(t; N) = Σ_{n≤N} a_n(F) / n^(1/2 + it)

def compute_dirichlet_series(coeffs, t_values, N):
 """
 Compute D_F(t; N) = Σ_{n=1}^N a_n / n^(1/2 + it) for array of t values
 Uses Kahan summation for numerical stability
 
 Parameters:
 - coeffs: array of coefficients a_n (length N+1, with coeffs[0] unused)
 - t_values: array of t values at which to evaluate
 - N: truncation length
 
 Returns:
 - Complex array of D_F(t; N) values
 """
 n = np.arange(1, N+1)
 n_sqrt = np.sqrt(n)
 
 results = np.zeros(len(t_values), dtype=complex)
 
 for idx, t in enumerate(t_values):
 # Compute n^(-1/2 - it) = n^(-1/2) * exp(-it * log(n))
 phase = np.exp(-1j * t * np.log(n))
 terms = coeffs[1:N+1] / n_sqrt * phase
 
 # Kahan summation for better numerical stability
 sum_val = 0.0 + 0.0j
 c = 0.0 + 0.0j
 for term in terms:
 y = term - c
 t_temp = sum_val + y
 c = (t_temp - sum_val) - y
 sum_val = t_temp
 
 results[idx] = sum_val
 
 return results

# Define coefficient functions
def get_zeta_coeffs(N):
 """Riemann zeta: a_n = 1 for all n"""
 return np.ones(N+1)

def get_chi4_coeffs(N):
 """Real character mod 5: chi_4(n)"""
 coeffs = np.zeros(N+1)
 for n in range(1, N+1):
 r = n % 5
 if r == 0:
 coeffs[n] = 0
 elif r == 1:
 coeffs[n] = 1
 elif r == 2:
 coeffs[n] = -1
 elif r == 3:
 coeffs[n] = -1
 elif r == 4:
 coeffs[n] = 1
 return coeffs

def get_liouville_coeffs(N):
 """Liouville function λ(n) = (-1)^Ω(n) where Ω(n) counts prime factors with multiplicity"""
 coeffs = np.ones(N+1)
 coeffs[0] = 0
 
 # Sieve to compute Ω(n)
 omega = np.zeros(N+1, dtype=int)
 for p in range(2, N+1):
 if omega[p] == 0: # p is prime
 # Mark all multiples
 for multiple in range(p, N+1, p):
 # Count powers of p dividing multiple
 temp = multiple
 while temp % p == 0:
 omega[multiple] += 1
 temp //= p
 
 for n in range(1, N+1):
 coeffs[n] = (-1) ** omega[n]
 
 return coeffs

def get_mobius_coeffs(N):
 """Möbius function μ(n)"""
 coeffs = np.zeros(N+1)
 coeffs[1] = 1
 
 # Sieve for μ(n)
 is_prime = np.ones(N+1, dtype=bool)
 is_prime[0] = is_prime[1] = False
 is_squarefree = np.ones(N+1, dtype=bool)
 num_prime_factors = np.zeros(N+1, dtype=int)
 
 for p in range(2, N+1):
 if is_prime[p]:
 # Mark non-primes
 for multiple in range(2*p, N+1, p):
 is_prime[multiple] = False
 num_prime_factors[multiple] += 1
 
 num_prime_factors[p] = 1
 
 # Mark non-squarefree numbers
 p_squared = p * p
 if p_squared <= N:
 for multiple in range(p_squared, N+1, p_squared):
 is_squarefree[multiple] = False
 
 for n in range(1, N+1):
 if not is_squarefree[n]:
 coeffs[n] = 0
 else:
 coeffs[n] = (-1) ** num_prime_factors[n]
 
 return coeffs

def get_random_multiplicative_coeffs(N, seed=42):
 """Random multiplicative function: a_p = ±1 uniformly, extended multiplicatively"""
 np.random.seed(seed)
 coeffs = np.zeros(N+1)
 coeffs[1] = 1
 
 # Find primes
 is_prime = np.ones(N+1, dtype=bool)
 is_prime[0] = is_prime[1] = False
 for p in range(2, int(np.sqrt(N))+1):
 if is_prime[p]:
 for multiple in range(p*p, N+1, p):
 is_prime[multiple] = False
 
 primes = [i for i in range(2, N+1) if is_prime[i]]
 
 # Assign random signs to primes
 prime_values = {}
 for p in primes:
 prime_values[p] = np.random.choice([-1, 1])
 
 # Extend multiplicatively
 for n in range(2, N+1):
 if is_prime[n]:
 coeffs[n] = prime_values[n]
 else:
 # Factor n and compute product
 temp_n = n
 value = 1
 for p in primes:
 if p * p > temp_n:
 break
 while temp_n % p == 0:
 value *= prime_values[p]
 temp_n //= p
 if temp_n > 1: # Remaining prime factor
 value *= prime_values[temp_n]
 coeffs[n] = value
 
 return coeffs

def get_fully_random_coeffs(N, seed=42):
 """Fully random coefficients: a_n = ±1 uniformly and independently"""
 np.random.seed(seed)
 return np.random.choice([-1, 1], size=N+1)

print("Coefficient functions defined.")
print("Testing coefficient generation for N=100...")

# Quick test
N_test = 100
zeta_test = get_zeta_coeffs(N_test)
chi4_test = get_chi4_coeffs(N_test)
print(f"ζ(s) coefficients [1-10]: {zeta_test[1:11]}")
print(f"L(s,χ₄) coefficients [1-10]: {chi4_test[1:11]}")


Coefficient functions defined.
Testing coefficient generation for N=100...
ζ(s) coefficients [1-10]: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
L(s,χ₄) coefficients [1-10]: [ 1. -1. -1. 1. 0. 1. -1. -1. 1. 0.]


In [3]:

# Now implement the validated historical L_DH coefficients
# Based on the dataset description, there are multiple versions. 
# We'll use the one with "correct real, periodic coefficients" (r78, r82)
# with CAS≈3.6, described as: a_n=2 for n≡1(mod 4), etc.

def get_validated_L_DH_coeffs(N):
 """
 Validated historical L_DH with real periodic coefficients
 Based on the Davenport-Heilbronn construction using character mod 5
 
 From the PDF: The DH function is defined as:
 L_DH(s) = (1-iκ)/2 * L(s,χ) + (1+iκ)/2 * L(s,χ̄)
 where χ is the complex character mod 5 of order 4
 
 χ(1)=1, χ(2)=i, χ(3)=-i, χ(4)=-1, χ(0)=0
 κ = (√5-1) / (2√(5(√5-1)))
 
 The coefficients are: a_n = (1-iκ)/2 * χ(n) + (1+iκ)/2 * χ̄(n)
 """
 coeffs = np.zeros(N+1, dtype=complex)
 
 # Define χ for mod 5
 chi_values = {0: 0, 1: 1, 2: 1j, 3: -1j, 4: -1}
 
 # Compute κ
 sqrt5 = np.sqrt(5)
 kappa = (sqrt5 - 1) / (2 * np.sqrt(5 * (sqrt5 - 1)))
 
 for n in range(1, N+1):
 r = n % 5
 chi_n = chi_values[r]
 chi_bar_n = np.conj(chi_n)
 
 # a_n = (1-iκ)/2 * χ(n) + (1+iκ)/2 * χ̄(n)
 coeffs[n] = (1 - 1j*kappa)/2 * chi_n + (1 + 1j*kappa)/2 * chi_bar_n
 
 return coeffs

# Test the L_DH coefficients
N_test = 20
L_DH_test = get_validated_L_DH_coeffs(N_test)
print("Validated L_DH coefficients [1-20]:")
for i in range(1, 21):
 print(f"a_{i} = {L_DH_test[i]:.4f}")

# Check if coefficients are mostly real (they should be according to description)
print(f"\nMax imaginary part: {np.max(np.abs(np.imag(L_DH_test[1:21]))):.6f}")
print(f"Max real part: {np.max(np.abs(np.real(L_DH_test[1:21]))):.6f}")


Validated L_DH coefficients [1-20]:
a_1 = 1.0000+0.0000j
a_2 = 0.2486+0.0000j
a_3 = -0.2486+0.0000j
a_4 = -1.0000+0.0000j
a_5 = 0.0000+0.0000j
a_6 = 1.0000+0.0000j
a_7 = 0.2486+0.0000j
a_8 = -0.2486+0.0000j
a_9 = -1.0000+0.0000j
a_10 = 0.0000+0.0000j
a_11 = 1.0000+0.0000j
a_12 = 0.2486+0.0000j
a_13 = -0.2486+0.0000j
a_14 = -1.0000+0.0000j
a_15 = 0.0000+0.0000j
a_16 = 1.0000+0.0000j
a_17 = 0.2486+0.0000j
a_18 = -0.2486+0.0000j
a_19 = -1.0000+0.0000j
a_20 = 0.0000+0.0000j

Max imaginary part: 0.000000
Max real part: 1.000000


In [4]:

# Good! The L_DH coefficients are real and periodic (period 5)
# Now let's set up the main computation parameters

# Parameters
N = 10**5 # Truncation length
t_min = 1000
t_max = 20000
dt = 2 * np.pi / np.log(N) # Standard sampling resolution

# Generate t values
t_values = np.arange(t_min, t_max, dt)
n_t_points = len(t_values)

print(f"Computation parameters:")
print(f" N = {N:,}")
print(f" t range: [{t_min}, {t_max}]")
print(f" dt = {dt:.4f}")
print(f" Number of t points: {n_t_points:,}")
print(f" Expected computation time estimate: ~{n_t_points * N / 1e9:.1f} seconds (rough)")

# This is a large computation. Let's start with a test on smaller parameters
# to verify the code works, then scale up

print("\n" + "="*60)
print("Starting with smaller test parameters to verify code...")
print("="*60)


Computation parameters:
 N = 100,000
 t range: [1000, 20000]
 dt = 0.5458
 Number of t points: 34,815
 Expected computation time estimate: ~3.5 seconds (rough)

Starting with smaller test parameters to verify code...


In [5]:

# Let's test with smaller N first to verify the pipeline works
# Then we'll scale up to N=10^5

N_test = 10**4
t_test = np.arange(1000, 1100, 0.5) # Small test range

print(f"Test computation: N={N_test:,}, t in [1000, 1100], {len(t_test)} points")

# Generate all coefficient arrays
print("\nGenerating coefficient arrays...")
coeffs_dict = {
 'zeta': get_zeta_coeffs(N_test),
 'chi4': get_chi4_coeffs(N_test),
 'liouville': get_liouville_coeffs(N_test),
 'mobius': get_mobius_coeffs(N_test),
 'f_rand': get_random_multiplicative_coeffs(N_test, seed=42),
 'f_fully_rand': get_fully_random_coeffs(N_test, seed=42),
 'L_DH': get_validated_L_DH_coeffs(N_test)
}

print("Coefficient arrays generated.")
print("\nComputing D_F(t; N) for test case...")

# Compute D_F for one function as test
D_zeta_test = compute_dirichlet_series(coeffs_dict['zeta'], t_test, N_test)

print(f"D_zeta computed: {len(D_zeta_test)} values")
print(f"Sample |D_zeta| values: {np.abs(D_zeta_test[:5])}")
print(f"Mean |D_zeta|: {np.mean(np.abs(D_zeta_test)):.4f}")
print(f"Max |D_zeta|: {np.max(np.abs(D_zeta_test)):.4f}")


Test computation: N=10,000, t in [1000, 1100], 200 points

Generating coefficient arrays...
Coefficient arrays generated.

Computing D_F(t; N) for test case...


D_zeta computed: 200 values
Sample |D_zeta| values: [1.04492621 2.49059952 1.60361031 0.63551289 0.87589195]
Mean |D_zeta|: 1.7971
Max |D_zeta|: 9.5306


In [6]:

# Good! The computation works. Now let's define the R_comp metric
# R_comp measures resonance/coherence in a sliding window

def compute_R_comp(D_F_values, window_size=50):
 """
 Compute R_comp metric using a sliding window.
 
 R_comp is defined as a measure of local coherence/resonance.
 Based on the dataset description, R_comp measures the relative 
 strength of local peaks.
 
 One common definition: R_comp(t) = |D_F(t)| / <|D_F|>_local
 where <|D_F|>_local is the local mean in a window around t
 
 Parameters:
 - D_F_values: complex array of D_F(t) values
 - window_size: size of sliding window for local statistics
 
 Returns:
 - R_comp: array of R_comp values
 """
 abs_D = np.abs(D_F_values)
 n = len(abs_D)
 R_comp = np.zeros(n)
 
 # Use sliding window to compute local mean
 half_window = window_size // 2
 
 for i in range(n):
 # Define window bounds
 start = max(0, i - half_window)
 end = min(n, i + half_window + 1)
 
 # Local statistics
 local_mean = np.mean(abs_D[start:end])
 
 # Avoid division by zero
 if local_mean > 0:
 R_comp[i] = abs_D[i] / local_mean
 else:
 R_comp[i] = 1.0
 
 return R_comp

# Test R_comp computation
R_comp_zeta_test = compute_R_comp(D_zeta_test, window_size=50)

print(f"R_comp computed: {len(R_comp_zeta_test)} values")
print(f"Sample R_comp values: {R_comp_zeta_test[:10]}")
print(f"Mean R_comp: {np.mean(R_comp_zeta_test):.4f}")
print(f"Max R_comp: {np.max(R_comp_zeta_test):.4f}")
print(f"Min R_comp: {np.min(R_comp_zeta_test):.4f}")

# Visualize test data
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(t_test, np.abs(D_zeta_test), linewidth=0.5)
axes[0].set_xlabel('t')
axes[0].set_ylabel('|D_F(t)|')
axes[0].set_title('Test: |D_ζ(t)| for N=10^4')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_test, R_comp_zeta_test, linewidth=0.5, color='orange')
axes[1].set_xlabel('t')
axes[1].set_ylabel('R_comp(t)')
axes[1].set_title('Test: R_comp(t) for ζ')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('test_R_comp.png', dpi=100, bbox_inches='tight')
plt.close()

print("\nTest plot saved as 'test_R_comp.png'")


R_comp computed: 200 values
Sample R_comp values: [0.57716516 1.42395962 0.90957531 0.34815272 0.48367381 0.08385901
 0.39643534 0.51022611 1.04587033 0.48615178]
Mean R_comp: 0.9881
Max R_comp: 5.1813
Min R_comp: 0.0307



Test plot saved as 'test_R_comp.png'


In [7]:

# Good! Now let's proceed with the full computation at N=10^5
# We'll compute for all 7 functions across the full t range

# Due to computational constraints, let's optimize the computation
# We can vectorize better and use fewer t points if needed

N = 10**5
t_min = 1000
t_max = 20000

# Use slightly larger dt to reduce computation time if needed
dt = 2 * np.pi / np.log(N) # Standard resolution
t_values = np.arange(t_min, t_max, dt)

print(f"Full computation parameters:")
print(f" N = {N:,}")
print(f" t range: [{t_min}, {t_max}]")
print(f" dt = {dt:.4f}")
print(f" Number of t points: {len(t_values):,}")

print("\nGenerating coefficient arrays for all functions (N=10^5)...")

# Generate coefficients
coeffs_full = {
 'zeta': get_zeta_coeffs(N),
 'chi4': get_chi4_coeffs(N),
 'liouville': get_liouville_coeffs(N),
 'mobius': get_mobius_coeffs(N),
 'f_rand': get_random_multiplicative_coeffs(N, seed=42),
 'f_fully_rand': get_fully_random_coeffs(N, seed=42),
 'L_DH': get_validated_L_DH_coeffs(N)
}

print("Coefficient arrays generated successfully.")
print("\nMemory check:")
for name, coeffs in coeffs_full.items():
 size_mb = coeffs.nbytes / (1024**2)
 print(f" {name:15s}: {size_mb:.2f} MB")


Full computation parameters:
 N = 100,000
 t range: [1000, 20000]
 dt = 0.5458
 Number of t points: 34,815

Generating coefficient arrays for all functions (N=10^5)...


Coefficient arrays generated successfully.

Memory check:
 zeta : 0.76 MB
 chi4 : 0.76 MB
 liouville : 0.76 MB
 mobius : 0.76 MB
 f_rand : 0.76 MB
 f_fully_rand : 0.76 MB
 L_DH : 1.53 MB


In [8]:

# Now compute D_F and R_comp for all functions
# This is the computationally intensive step

import time

# Store results
D_F_results = {}
R_comp_results = {}

# Optimized computation function
def compute_dirichlet_series_optimized(coeffs, t_values, N, batch_size=1000):
 """
 Compute D_F(t; N) in batches for memory efficiency
 """
 n = np.arange(1, N+1)
 n_sqrt = np.sqrt(n)
 log_n = np.log(n)
 
 n_t_points = len(t_values)
 results = np.zeros(n_t_points, dtype=complex)
 
 # Process in batches
 for batch_start in range(0, n_t_points, batch_size):
 batch_end = min(batch_start + batch_size, n_t_points)
 t_batch = t_values[batch_start:batch_end]
 
 # Vectorized computation for this batch
 # D_F(t) = Σ_n a_n / n^(1/2) * exp(-i t log(n))
 # Shape: (len(t_batch), N)
 t_log_n = np.outer(t_batch, log_n) # t * log(n) for all combinations
 phases = np.exp(-1j * t_log_n)
 
 # Multiply by a_n / sqrt(n)
 coeffs_normalized = coeffs[1:N+1] / n_sqrt
 
 # Sum over n for each t
 results[batch_start:batch_end] = np.dot(phases, coeffs_normalized)
 
 return results

print("Computing D_F(t; N) and R_comp(t) for all functions...")
print("This may take several minutes...\n")

for func_name, coeffs in coeffs_full.items():
 print(f"Processing {func_name}...", end=' ', flush=True)
 start_time = time.time()
 
 # Compute D_F
 D_F = compute_dirichlet_series_optimized(coeffs, t_values, N, batch_size=1000)
 D_F_results[func_name] = D_F
 
 # Compute R_comp
 R_comp = compute_R_comp(D_F, window_size=100)
 R_comp_results[func_name] = R_comp
 
 elapsed = time.time() - start_time
 print(f"Done in {elapsed:.1f}s (mean |D_F|={np.mean(np.abs(D_F)):.3f}, mean R_comp={np.mean(R_comp):.3f})")

print("\nAll computations complete!")


Computing D_F(t; N) and R_comp(t) for all functions...
This may take several minutes...

Processing zeta... 

Done in 142.9s (mean |D_F|=1.916, mean R_comp=1.002)
Processing chi4... 

Done in 144.7s (mean |D_F|=1.902, mean R_comp=1.003)
Processing liouville... 

Done in 143.6s (mean |D_F|=2.189, mean R_comp=1.002)
Processing mobius... 

Done in 142.7s (mean |D_F|=2.010, mean R_comp=1.002)
Processing f_rand... 

Done in 142.7s (mean |D_F|=2.183, mean R_comp=1.002)
Processing f_fully_rand... 

Done in 142.9s (mean |D_F|=3.087, mean R_comp=1.000)
Processing L_DH... 

Done in 142.7s (mean |D_F|=1.685, mean R_comp=1.002)

All computations complete!


In [9]:

# Excellent! Now we have R_comp time series for all 7 functions
# Next step: Extract block maxima using 200 blocks

def extract_block_maxima(data, n_blocks=200):
 """
 Extract block maxima from time series data
 Divide data into n_blocks equal-sized blocks and take maximum from each
 
 Parameters:
 - data: array of values
 - n_blocks: number of blocks to divide data into
 
 Returns:
 - block_maxima: array of maximum values from each block
 """
 n = len(data)
 block_size = n // n_blocks
 
 block_maxima = []
 for i in range(n_blocks):
 start = i * block_size
 end = start + block_size if i < n_blocks - 1 else n
 block_max = np.max(data[start:end])
 block_maxima.append(block_max)
 
 return np.array(block_maxima)

# Extract block maxima for all functions
n_blocks = 200
block_maxima_dict = {}

print(f"Extracting block maxima ({n_blocks} blocks per function)...\n")

for func_name, R_comp in R_comp_results.items():
 block_max = extract_block_maxima(R_comp, n_blocks=n_blocks)
 block_maxima_dict[func_name] = block_max
 
 print(f"{func_name:15s}: mean={np.mean(block_max):.4f}, max={np.max(block_max):.4f}, min={np.min(block_max):.4f}")

print("\nBlock maxima extraction complete!")


Extracting block maxima (200 blocks per function)...

zeta : mean=7.0140, max=9.7601, min=4.7101
chi4 : mean=7.5608, max=10.9488, min=5.2577
liouville : mean=9.0720, max=16.5256, min=4.8600
mobius : mean=6.5134, max=11.6773, min=3.7645
f_rand : mean=9.2701, max=18.3878, min=4.7298
f_fully_rand : mean=2.6786, max=3.3631, min=2.0859
L_DH : mean=5.0265, max=7.5650, min=3.4519

Block maxima extraction complete!


In [10]:

# Now fit GEV distribution to block maxima for each function
# GEV has three parameters: location (μ), scale (σ), and shape (ξ)

from scipy.stats import genextreme

def fit_gev_with_bootstrap(data, n_bootstrap=1000, ci_level=0.95):
 """
 Fit GEV distribution to data and estimate parameters with bootstrap CI
 
 Parameters:
 - data: block maxima data
 - n_bootstrap: number of bootstrap samples
 - ci_level: confidence interval level
 
 Returns:
 - params: (shape, loc, scale) from MLE fit
 - ci: confidence intervals for each parameter
 """
 # Fit GEV using MLE
 # Note: scipy's genextreme uses shape parameter c = -ξ (negative of standard)
 shape, loc, scale = genextreme.fit(data)
 
 # Bootstrap for confidence intervals
 bootstrap_params = []
 n_data = len(data)
 
 for _ in range(n_bootstrap):
 # Resample with replacement
 bootstrap_sample = np.random.choice(data, size=n_data, replace=True)
 try:
 bs_shape, bs_loc, bs_scale = genextreme.fit(bootstrap_sample)
 bootstrap_params.append([bs_shape, bs_loc, bs_scale])
 except:
 # Skip failed fits
 continue
 
 bootstrap_params = np.array(bootstrap_params)
 
 # Compute confidence intervals
 alpha = 1 - ci_level
 ci_lower = np.percentile(bootstrap_params, 100 * alpha/2, axis=0)
 ci_upper = np.percentile(bootstrap_params, 100 * (1 - alpha/2), axis=0)
 
 ci = np.column_stack([ci_lower, ci_upper])
 
 return (shape, loc, scale), ci, bootstrap_params

# Fit GEV to all functions
print("Fitting GEV distributions with bootstrap (1000 samples)...")
print("This may take a few minutes...\n")

gev_results = {}

for func_name, block_max in block_maxima_dict.items():
 print(f"Fitting {func_name}...", end=' ', flush=True)
 start_time = time.time()
 
 params, ci, bootstrap_params = fit_gev_with_bootstrap(block_max, n_bootstrap=1000)
 shape_c, loc, scale = params
 
 # Convert scipy's c parameter to standard ξ (shape parameter)
 # scipy uses c = -ξ
 xi = -shape_c
 xi_ci = -ci[0][::-1] # Reverse order because of negation
 
 gev_results[func_name] = {
 'xi': xi,
 'xi_ci': xi_ci,
 'loc': loc,
 'scale': scale,
 'params_full': params,
 'ci_full': ci,
 'bootstrap_params': bootstrap_params
 }
 
 elapsed = time.time() - start_time
 print(f"Done in {elapsed:.1f}s (ξ={xi:.4f}, CI=[{xi_ci[0]:.4f}, {xi_ci[1]:.4f}])")

print("\nGEV fitting complete!")


Fitting GEV distributions with bootstrap (1000 samples)...
This may take a few minutes...

Fitting zeta... 

Done in 51.1s (ξ=-0.2030, CI=[-0.2777, -0.1293])
Fitting chi4... 

Done in 49.3s (ξ=-0.0950, CI=[-0.1764, -0.0017])
Fitting liouville... 

Done in 47.0s (ξ=0.0622, CI=[-0.0330, 0.1536])
Fitting mobius... 

Done in 45.8s (ξ=-0.0598, CI=[-0.1586, 0.0291])
Fitting f_rand... 

Done in 48.7s (ξ=0.0200, CI=[-0.0973, 0.1281])
Fitting f_fully_rand... 

Done in 61.9s (ξ=-0.1492, CI=[-0.2409, -0.0443])
Fitting L_DH... 

Done in 51.0s (ξ=-0.1725, CI=[-0.3303, -0.0972])

GEV fitting complete!


In [11]:

# Create a summary table of the results
# Organize by the three classes

# Define classes
class_mapping = {
 'Standard': ['zeta', 'chi4'],
 'Anomalous-Resonant': ['liouville', 'mobius', 'L_DH'],
 'Control': ['f_rand', 'f_fully_rand']
}

# Create summary dataframe
summary_data = []

for class_name, func_names in class_mapping.items():
 for func_name in func_names:
 result = gev_results[func_name]
 summary_data.append({
 'Class': class_name,
 'Function': func_name,
 'ξ': result['xi'],
 'ξ_lower': result['xi_ci'][0],
 'ξ_upper': result['xi_ci'][1],
 'μ (location)': result['loc'],
 'σ (scale)': result['scale']
 })

summary_df = pd.DataFrame(summary_data)

print("="*80)
print("SUMMARY: GEV Shape Parameter (ξ) Estimates by Function Class")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)

# Calculate class statistics
print("\nCLASS-LEVEL STATISTICS:")
print("-"*80)

for class_name in ['Standard', 'Anomalous-Resonant', 'Control']:
 class_data = summary_df[summary_df['Class'] == class_name]
 xi_values = class_data['ξ'].values
 
 print(f"\n{class_name}:")
 print(f" Mean ξ: {np.mean(xi_values):.4f}")
 print(f" Std ξ: {np.std(xi_values, ddof=1):.4f}")
 print(f" Range: [{np.min(xi_values):.4f}, {np.max(xi_values):.4f}]")
 print(f" Functions: {', '.join(class_data['Function'].values)}")

print("-"*80)


SUMMARY: GEV Shape Parameter (ξ) Estimates by Function Class
 Class Function ξ ξ_lower ξ_upper μ (location) σ (scale)
 Standard zeta -0.202976 -0.277709 -0.129294 6.615182 0.975412
 Standard chi4 -0.094983 -0.176365 -0.001662 7.067928 0.993925
Anomalous-Resonant liouville 0.062219 -0.033015 0.153584 7.799910 1.974213
Anomalous-Resonant mobius -0.059753 -0.158649 0.029081 5.907879 1.157522
Anomalous-Resonant L_DH -0.172462 -0.330347 -0.097207 4.740731 0.665494
 Control f_rand 0.019982 -0.097294 0.128072 8.054053 2.021813
 Control f_fully_rand -0.149244 -0.240941 -0.044331 2.571719 0.236159

CLASS-LEVEL STATISTICS:
--------------------------------------------------------------------------------

Standard:
 Mean ξ: -0.1490
 Std ξ: 0.0764
 Range: [-0.2030, -0.0950]
 Functions: zeta, chi4

Anomalous-Resonant:
 Mean ξ: -0.0567
 Std ξ: 0.1174
 Range: [-0.1725, 0.0622]
 Functions: liouville, mobius, L_DH

Control:
 Mean ξ: -0.0646
 Std ξ: 0.1197
 Range: [-0.1492, 0.0200]
 Functions: f_rand, f_

In [12]:

# Now perform statistical tests: ANOVA and post-hoc comparisons

from scipy import stats as sp_stats
from scipy.stats import f_oneway
from itertools import combinations

# Prepare data for ANOVA (ξ values by class)
xi_by_class = {
 'Standard': summary_df[summary_df['Class'] == 'Standard']['ξ'].values,
 'Anomalous-Resonant': summary_df[summary_df['Class'] == 'Anomalous-Resonant']['ξ'].values,
 'Control': summary_df[summary_df['Class'] == 'Control']['ξ'].values
}

print("="*80)
print("STATISTICAL ANALYSIS: Testing for Differences Between Classes")
print("="*80)

# ANOVA
print("\n1. ONE-WAY ANOVA")
print("-"*80)

F_stat, p_value = f_oneway(
 xi_by_class['Standard'],
 xi_by_class['Anomalous-Resonant'],
 xi_by_class['Control']
)

print(f"F-statistic: {F_stat:.4f}")
print(f"p-value: {p_value:.4f}")

if p_value < 0.05:
 print(f"Result: Significant difference between classes (p < 0.05)")
else:
 print(f"Result: No significant difference between classes (p >= 0.05)")

# Post-hoc pairwise comparisons using t-tests
print("\n2. POST-HOC PAIRWISE COMPARISONS (Welch's t-test)")
print("-"*80)

class_names = ['Standard', 'Anomalous-Resonant', 'Control']
pairwise_results = []

for class1, class2 in combinations(class_names, 2):
 data1 = xi_by_class[class1]
 data2 = xi_by_class[class2]
 
 # Welch's t-test (does not assume equal variances)
 t_stat, p_val = sp_stats.ttest_ind(data1, data2, equal_var=False)
 
 mean_diff = np.mean(data1) - np.mean(data2)
 
 pairwise_results.append({
 'Comparison': f"{class1} vs {class2}",
 't-statistic': t_stat,
 'p-value': p_val,
 'Mean Difference': mean_diff
 })
 
 print(f"\n{class1} vs {class2}:")
 print(f" Mean ξ difference: {mean_diff:.4f}")
 print(f" t-statistic: {t_stat:.4f}")
 print(f" p-value: {p_val:.4f}")
 if p_val < 0.05:
 print(f" Result: Significant difference (p < 0.05)")
 else:
 print(f" Result: No significant difference (p >= 0.05)")

print("\n" + "="*80)

# Create pairwise comparison dataframe
pairwise_df = pd.DataFrame(pairwise_results)


STATISTICAL ANALYSIS: Testing for Differences Between Classes

1. ONE-WAY ANOVA
--------------------------------------------------------------------------------
F-statistic: 0.4790
p-value: 0.6509
Result: No significant difference between classes (p >= 0.05)

2. POST-HOC PAIRWISE COMPARISONS (Welch's t-test)
--------------------------------------------------------------------------------

Standard vs Anomalous-Resonant:
 Mean ξ difference: -0.0923
 t-statistic: -1.0654
 p-value: 0.3658
 Result: No significant difference (p >= 0.05)

Standard vs Control:
 Mean ξ difference: -0.0843
 t-statistic: -0.8403
 p-value: 0.5023
 Result: No significant difference (p >= 0.05)

Anomalous-Resonant vs Control:
 Mean ξ difference: 0.0080
 t-statistic: 0.0735
 p-value: 0.9475
 Result: No significant difference (p >= 0.05)



In [13]:

# Create grouped boxplot visualization of ξ estimates

fig, ax = plt.subplots(figsize=(12, 8))

# Prepare data for boxplot
plot_data = []
plot_labels = []
plot_colors = []

color_map = {
 'Standard': '#2E86AB',
 'Anomalous-Resonant': '#A23B72',
 'Control': '#F18F01'
}

for class_name in ['Standard', 'Anomalous-Resonant', 'Control']:
 class_data = summary_df[summary_df['Class'] == class_name]
 
 for _, row in class_data.iterrows():
 # Plot each function with its CI as error bars
 func_name = row['Function']
 xi = row['ξ']
 xi_lower = row['ξ_lower']
 xi_upper = row['ξ_upper']
 
 plot_data.append([xi_lower, xi, xi_upper])
 plot_labels.append(f"{func_name}\n({class_name})")
 plot_colors.append(color_map[class_name])

# Create boxplot with individual points
positions = np.arange(len(plot_labels))

# Plot error bars (CIs)
for i, (data, color) in enumerate(zip(plot_data, plot_colors)):
 xi_lower, xi, xi_upper = data
 ax.errorbar(i, xi, yerr=[[xi - xi_lower], [xi_upper - xi]], 
 fmt='o', markersize=10, capsize=8, capthick=2,
 color=color, ecolor=color, linewidth=2, alpha=0.8,
 label=plot_labels[i].split('\n')[1][1:-1] if i < 3 else None)

# Add horizontal line at ξ=0
ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='ξ=0')

# Styling
ax.set_xticks(positions)
ax.set_xticklabels([label.split('\n')[0] for label in plot_labels], rotation=45, ha='right')
ax.set_ylabel('GEV Shape Parameter (ξ)', fontsize=12, fontweight='bold')
ax.set_title('GEV Shape Parameter (ξ) Estimates with 95% Confidence Intervals\nby Function Class', 
 fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, axis='y')

# Add class separators
class_boundaries = [1.5, 4.5]
for boundary in class_boundaries:
 ax.axvline(x=boundary, color='black', linestyle='-', linewidth=1.5, alpha=0.3)

# Add class labels as text annotations
ax.text(0.5, ax.get_ylim()[1]*0.95, 'Standard', ha='center', fontsize=11, 
 fontweight='bold', color=color_map['Standard'])
ax.text(3, ax.get_ylim()[1]*0.95, 'Anomalous-Resonant', ha='center', fontsize=11,
 fontweight='bold', color=color_map['Anomalous-Resonant'])
ax.text(5.5, ax.get_ylim()[1]*0.95, 'Control', ha='center', fontsize=11,
 fontweight='bold', color=color_map['Control'])

# Custom legend for classes
from matplotlib.patches import Patch
legend_elements = [
 Patch(facecolor=color_map['Standard'], label='Standard'),
 Patch(facecolor=color_map['Anomalous-Resonant'], label='Anomalous-Resonant'),
 Patch(facecolor=color_map['Control'], label='Control')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('gev_xi_by_class.png', dpi=300, bbox_inches='tight')
plt.close()

print("Grouped visualization saved as 'gev_xi_by_class.png'")


Grouped visualization saved as 'gev_xi_by_class.png'


In [14]:

# Check for overlap in confidence intervals - a more direct test of distinctness

print("="*80)
print("CONFIDENCE INTERVAL OVERLAP ANALYSIS")
print("="*80)

# Function to check if two CIs overlap
def check_overlap(ci1, ci2):
 """Check if two confidence intervals overlap"""
 return not (ci1[1] < ci2[0] or ci2[1] < ci1[0])

# Check all pairwise overlaps
print("\nPairwise CI Overlap Check:")
print("-"*80)

func_names = summary_df['Function'].values
overlap_matrix = []

for i, func1 in enumerate(func_names):
 row1 = summary_df[summary_df['Function'] == func1].iloc[0]
 ci1 = (row1['ξ_lower'], row1['ξ_upper'])
 class1 = row1['Class']
 
 for j, func2 in enumerate(func_names):
 if i >= j:
 continue
 
 row2 = summary_df[summary_df['Function'] == func2].iloc[0]
 ci2 = (row2['ξ_lower'], row2['ξ_upper'])
 class2 = row2['Class']
 
 overlaps = check_overlap(ci1, ci2)
 
 print(f"{func1:15s} [{ci1[0]:.4f}, {ci1[1]:.4f}] vs", end=' ')
 print(f"{func2:15s} [{ci2[0]:.4f}, {ci2[1]:.4f}]: ", end='')
 
 if overlaps:
 print("OVERLAP")
 else:
 print("NO OVERLAP (distinct)")
 
 overlap_matrix.append({
 'Function 1': func1,
 'Class 1': class1,
 'Function 2': func2,
 'Class 2': class2,
 'Overlap': overlaps
 })

print("\n" + "="*80)

# Count overlaps within and between classes
overlap_df = pd.DataFrame(overlap_matrix)

print("\nOVERLAP SUMMARY:")
print("-"*80)

# Within-class overlaps
for class_name in ['Standard', 'Anomalous-Resonant', 'Control']:
 within_class = overlap_df[
 (overlap_df['Class 1'] == class_name) & 
 (overlap_df['Class 2'] == class_name)
 ]
 if len(within_class) > 0:
 n_overlap = within_class['Overlap'].sum()
 n_total = len(within_class)
 print(f"Within {class_name:20s}: {n_overlap}/{n_total} pairs overlap")

# Between-class overlaps
print("\nBetween classes:")
for class1, class2 in combinations(['Standard', 'Anomalous-Resonant', 'Control'], 2):
 between_class = overlap_df[
 ((overlap_df['Class 1'] == class1) & (overlap_df['Class 2'] == class2)) |
 ((overlap_df['Class 1'] == class2) & (overlap_df['Class 2'] == class1))
 ]
 if len(between_class) > 0:
 n_overlap = between_class['Overlap'].sum()
 n_total = len(between_class)
 print(f"{class1:20s} vs {class2:20s}: {n_overlap}/{n_total} pairs overlap")

print("="*80)


CONFIDENCE INTERVAL OVERLAP ANALYSIS

Pairwise CI Overlap Check:
--------------------------------------------------------------------------------
zeta [-0.2777, -0.1293] vs chi4 [-0.1764, -0.0017]: OVERLAP
zeta [-0.2777, -0.1293] vs liouville [-0.0330, 0.1536]: NO OVERLAP (distinct)
zeta [-0.2777, -0.1293] vs mobius [-0.1586, 0.0291]: OVERLAP
zeta [-0.2777, -0.1293] vs L_DH [-0.3303, -0.0972]: OVERLAP
zeta [-0.2777, -0.1293] vs f_rand [-0.0973, 0.1281]: NO OVERLAP (distinct)
zeta [-0.2777, -0.1293] vs f_fully_rand [-0.2409, -0.0443]: OVERLAP
chi4 [-0.1764, -0.0017] vs liouville [-0.0330, 0.1536]: OVERLAP
chi4 [-0.1764, -0.0017] vs mobius [-0.1586, 0.0291]: OVERLAP
chi4 [-0.1764, -0.0017] vs L_DH [-0.3303, -0.0972]: OVERLAP
chi4 [-0.1764, -0.0017] vs f_rand [-0.0973, 0.1281]: OVERLAP
chi4 [-0.1764, -0.0017] vs f_fully_rand [-0.2409, -0.0443]: OVERLAP
liouville [-0.0330, 0.1536] vs mobius [-0.1586, 0.0291]: OVERLAP
liouville [-0.0330, 0.1536] vs L_DH [-0.3303, -0.0972]: NO OVERLAP (disti

In [15]:

# Let's also examine the distribution of block maxima for each function
# to visualize the differences

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

func_names = ['zeta', 'chi4', 'liouville', 'mobius', 'f_rand', 'f_fully_rand', 'L_DH']
class_names_map = {
 'zeta': 'Standard',
 'chi4': 'Standard',
 'liouville': 'Anomalous-Resonant',
 'mobius': 'Anomalous-Resonant',
 'L_DH': 'Anomalous-Resonant',
 'f_rand': 'Control',
 'f_fully_rand': 'Control'
}

color_map = {
 'Standard': '#2E86AB',
 'Anomalous-Resonant': '#A23B72',
 'Control': '#F18F01'
}

for idx, func_name in enumerate(func_names):
 ax = axes[idx]
 block_max = block_maxima_dict[func_name]
 class_name = class_names_map[func_name]
 color = color_map[class_name]
 
 # Histogram
 ax.hist(block_max, bins=30, density=True, alpha=0.6, color=color, edgecolor='black')
 
 # Fit GEV and plot
 result = gev_results[func_name]
 shape_c = -result['xi'] # Convert back to scipy convention
 loc = result['loc']
 scale = result['scale']
 
 x = np.linspace(block_max.min(), block_max.max(), 100)
 pdf = genextreme.pdf(x, shape_c, loc=loc, scale=scale)
 ax.plot(x, pdf, 'r-', linewidth=2, label=f'GEV fit')
 
 # Add ξ value to plot
 ax.text(0.05, 0.95, f'ξ = {result["xi"]:.3f}', transform=ax.transAxes,
 fontsize=10, verticalalignment='top',
 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
 
 ax.set_title(f'{func_name}\n({class_name})', fontweight='bold')
 ax.set_xlabel('Block Maximum R_comp')
 ax.set_ylabel('Density')
 ax.grid(True, alpha=0.3)
 ax.legend(fontsize=8)

# Hide the last subplot (8th)
axes[7].axis('off')

plt.suptitle('Distribution of Block Maxima (R_comp) with GEV Fits', 
 fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('block_maxima_distributions.png', dpi=300, bbox_inches='tight')
plt.close()

print("Block maxima distributions plot saved as 'block_maxima_distributions.png'")


Block maxima distributions plot saved as 'block_maxima_distributions.png'


In [16]:

# Additional analysis: Effect sizes and practical significance

print("="*80)
print("EFFECT SIZE ANALYSIS")
print("="*80)

# Cohen's d for pairwise comparisons
def cohens_d(group1, group2):
 """Calculate Cohen's d effect size"""
 n1, n2 = len(group1), len(group2)
 var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
 pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
 return (np.mean(group1) - np.mean(group2)) / pooled_std if pooled_std > 0 else 0

print("\nCohen's d Effect Sizes (for ξ values):")
print("-"*80)

class_names = ['Standard', 'Anomalous-Resonant', 'Control']

for class1, class2 in combinations(class_names, 2):
 data1 = xi_by_class[class1]
 data2 = xi_by_class[class2]
 
 d = cohens_d(data1, data2)
 
 # Interpret Cohen's d
 if abs(d) < 0.2:
 interpretation = "negligible"
 elif abs(d) < 0.5:
 interpretation = "small"
 elif abs(d) < 0.8:
 interpretation = "medium"
 else:
 interpretation = "large"
 
 print(f"{class1:25s} vs {class2:25s}: d = {d:6.3f} ({interpretation})")

print("\n" + "="*80)

# Summary statistics for block maxima (not just ξ)
print("\nBLOCK MAXIMA SUMMARY STATISTICS:")
print("-"*80)

block_max_stats = []

for func_name in func_names:
 block_max = block_maxima_dict[func_name]
 class_name = class_names_map[func_name]
 
 block_max_stats.append({
 'Function': func_name,
 'Class': class_name,
 'Mean': np.mean(block_max),
 'Std': np.std(block_max, ddof=1),
 'Min': np.min(block_max),
 'Max': np.max(block_max),
 'Q25': np.percentile(block_max, 25),
 'Q50': np.percentile(block_max, 50),
 'Q75': np.percentile(block_max, 75)
 })

block_max_df = pd.DataFrame(block_max_stats)
print(block_max_df.to_string(index=False))
print("="*80)


EFFECT SIZE ANALYSIS

Cohen's d Effect Sizes (for ξ values):
--------------------------------------------------------------------------------
Standard vs Anomalous-Resonant : d = -0.875 (large)
Standard vs Control : d = -0.840 (large)
Anomalous-Resonant vs Control : d = 0.067 (negligible)


BLOCK MAXIMA SUMMARY STATISTICS:
--------------------------------------------------------------------------------
 Function Class Mean Std Min Max Q25 Q50 Q75
 zeta Standard 7.013997 1.031429 4.710082 9.760087 6.350914 6.948070 7.705770
 chi4 Standard 7.560840 1.151257 5.257740 10.948832 6.786989 7.402208 8.332742
 liouville Anomalous-Resonant 9.072041 2.660885 4.860010 16.525617 7.102341 8.499081 10.465780
 mobius Anomalous-Resonant 6.513362 1.387362 3.764457 11.677335 5.509746 6.377815 7.273371
 f_rand Control 9.270104 2.609878 4.729829 18.387818 7.174710 8.970411 10.679650
f_fully_rand Control 2.678628 0.262196 2.085870 3.363118 2.470474 2.661356 2.838285
 L_DH Anomalous-Resonant 5.026520 0.71236

In [17]:

# Create final summary figure for the paper
# This will be a single column figure with two panels stacked vertically

fig = plt.figure(figsize=(10, 12))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.3)

# Panel A: ξ estimates with CIs by class
ax1 = fig.add_subplot(gs[0])

positions = np.arange(len(func_names))
colors_list = [color_map[class_names_map[f]] for f in func_names]

for i, func_name in enumerate(func_names):
 result = gev_results[func_name]
 xi = result['xi']
 xi_ci = result['xi_ci']
 color = colors_list[i]
 
 ax1.errorbar(i, xi, 
 yerr=[[xi - xi_ci[0]], [xi_ci[1] - xi]], 
 fmt='o', markersize=10, capsize=8, capthick=2,
 color=color, ecolor=color, linewidth=2, alpha=0.8)

ax1.axhline(y=0, color='gray', linestyle='--', linewidth=1.5, alpha=0.5, label='ξ=0')
ax1.set_xticks(positions)
ax1.set_xticklabels(func_names, rotation=45, ha='right')
ax1.set_ylabel('GEV Shape Parameter (ξ)', fontsize=12, fontweight='bold')
ax1.set_title('A. GEV Shape Parameter Estimates with 95% Confidence Intervals', 
 fontsize=13, fontweight='bold', loc='left', pad=10)
ax1.grid(True, alpha=0.3, axis='y')

# Add class separators
ax1.axvline(x=1.5, color='black', linestyle='-', linewidth=1.5, alpha=0.3)
ax1.axvline(x=4.5, color='black', linestyle='-', linewidth=1.5, alpha=0.3)

# Add class labels
y_pos = ax1.get_ylim()[1] * 0.92
ax1.text(0.5, y_pos, 'Standard', ha='center', fontsize=10, 
 fontweight='bold', color=color_map['Standard'])
ax1.text(3, y_pos, 'Anomalous-Resonant', ha='center', fontsize=10,
 fontweight='bold', color=color_map['Anomalous-Resonant'])
ax1.text(5.5, y_pos, 'Control', ha='center', fontsize=10,
 fontweight='bold', color=color_map['Control'])

# Panel B: Class means with error bars
ax2 = fig.add_subplot(gs[1])

class_means = []
class_stds = []
class_labels = []
class_colors = []

for class_name in ['Standard', 'Anomalous-Resonant', 'Control']:
 xi_values = xi_by_class[class_name]
 class_means.append(np.mean(xi_values))
 class_stds.append(np.std(xi_values, ddof=1))
 class_labels.append(class_name)
 class_colors.append(color_map[class_name])

x_pos = np.arange(len(class_labels))

bars = ax2.bar(x_pos, class_means, yerr=class_stds, 
 color=class_colors, alpha=0.7, 
 capsize=10, ecolor='black', linewidth=2,
 edgecolor='black', width=0.6)

ax2.axhline(y=0, color='gray', linestyle='--', linewidth=1.5, alpha=0.5)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(class_labels, fontsize=11)
ax2.set_ylabel('Mean ξ ± SD', fontsize=12, fontweight='bold')
ax2.set_title('B. Mean GEV Shape Parameter by Function Class', 
 fontsize=13, fontweight='bold', loc='left', pad=10)
ax2.grid(True, alpha=0.3, axis='y')

# Add p-value annotation from ANOVA
ax2.text(0.98, 0.95, f'ANOVA: F={F_stat:.3f}, p={p_value:.3f}', 
 transform=ax2.transAxes,
 fontsize=10, verticalalignment='top', horizontalalignment='right',
 bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'))

# Add individual data points
for i, class_name in enumerate(['Standard', 'Anomalous-Resonant', 'Control']):
 xi_values = xi_by_class[class_name]
 x_jitter = np.random.normal(i, 0.04, size=len(xi_values))
 ax2.scatter(x_jitter, xi_values, color='black', s=80, alpha=0.6, zorder=3)

plt.tight_layout()
plt.savefig('final_figure_gev_analysis.png', dpi=300, bbox_inches='tight')
plt.close()

print("Final summary figure saved as 'final_figure_gev_analysis.png'")


Final summary figure saved as 'final_figure_gev_analysis.png'


In [18]:

# Save results to CSV files for future reference

# Save main results
summary_df.to_csv('gev_shape_parameter_results.csv', index=False)
print("Main results saved to 'gev_shape_parameter_results.csv'")

# Save pairwise comparison results
pairwise_df.to_csv('pairwise_comparisons.csv', index=False)
print("Pairwise comparisons saved to 'pairwise_comparisons.csv'")

# Save block maxima statistics
block_max_df.to_csv('block_maxima_statistics.csv', index=False)
print("Block maxima statistics saved to 'block_maxima_statistics.csv'")

# Save detailed ANOVA results
anova_results = {
 'Test': ['One-Way ANOVA'],
 'F-statistic': [F_stat],
 'p-value': [p_value],
 'Conclusion': ['No significant difference' if p_value >= 0.05 else 'Significant difference']
}
anova_df = pd.DataFrame(anova_results)
anova_df.to_csv('anova_results.csv', index=False)
print("ANOVA results saved to 'anova_results.csv'")

print("\nAll results saved successfully!")


Main results saved to 'gev_shape_parameter_results.csv'
Pairwise comparisons saved to 'pairwise_comparisons.csv'
Block maxima statistics saved to 'block_maxima_statistics.csv'
ANOVA results saved to 'anova_results.csv'

All results saved successfully!


In [19]:

# Final comprehensive summary and interpretation

print("="*80)
print("COMPREHENSIVE ANALYSIS SUMMARY")
print("="*80)

print("\n1. HYPOTHESIS TESTING")
print("-"*80)
print("Hypothesis: The 'Anomalous-Resonant' class would have a GEV shape parameter")
print("ξ that is statistically distinct from both 'Standard' and 'Control' classes.")
print()
print("Result: HYPOTHESIS NOT SUPPORTED")
print()
print(f"- One-way ANOVA: F = {F_stat:.4f}, p = {p_value:.4f} (not significant)")
print(f"- No significant differences between any pair of classes (all p > 0.05)")
print()

print("\n2. KEY FINDINGS")
print("-"*80)

print("\nGEV Shape Parameter (ξ) by Class:")
for class_name in ['Standard', 'Anomalous-Resonant', 'Control']:
 xi_vals = xi_by_class[class_name]
 print(f" {class_name:25s}: Mean = {np.mean(xi_vals):6.3f}, SD = {np.std(xi_vals, ddof=1):6.3f}")

print("\n\nIndividual Function Results:")
for _, row in summary_df.iterrows():
 print(f" {row['Function']:15s} ({row['Class']:20s}): ", end='')
 print(f"ξ = {row['ξ']:7.4f} [{row['ξ_lower']:7.4f}, {row['ξ_upper']:7.4f}]")

print("\n\n3. CONFIDENCE INTERVAL OVERLAPS")
print("-"*80)
print("Key observations:")
print(" - All 'Standard' class CIs overlap with most other functions")
print(" - 'Anomalous-Resonant' class shows heterogeneity:")
print(" * L(s,λ(n)) [liouville] has positive ξ (≈0.062)")
print(" * L(s,μ(n)) [mobius] and L_DH have negative ξ")
print(" - No clear separation between classes")
print()
print("Non-overlapping pairs (potentially distinct):")
non_overlap_count = 0
for _, row in overlap_df.iterrows():
 if not row['Overlap']:
 non_overlap_count += 1
 print(f" - {row['Function 1']:15s} vs {row['Function 2']:15s}", end='')
 print(f" ({row['Class 1']:20s} vs {row['Class 2']:20s})")

print(f"\nTotal: {non_overlap_count} out of {len(overlap_df)} pairs show no overlap")

print("\n\n4. EFFECT SIZES")
print("-"*80)
print("Cohen's d between classes:")
for class1, class2 in combinations(['Standard', 'Anomalous-Resonant', 'Control'], 2):
 d = cohens_d(xi_by_class[class1], xi_by_class[class2])
 interpretation = "large" if abs(d) >= 0.8 else "medium" if abs(d) >= 0.5 else "small" if abs(d) >= 0.2 else "negligible"
 print(f" {class1:25s} vs {class2:25s}: d = {d:6.3f} ({interpretation})")

print("\n\n5. INTERPRETATION")
print("-"*80)
print("The analysis reveals that extreme value distributions of R_comp metrics do NOT")
print("show clear separation between Standard, Anomalous-Resonant, and Control classes.")
print()
print("Key issues:")
print(" 1. High within-class heterogeneity, especially in Anomalous-Resonant class")
print(" 2. Functions within the same class show diverse tail behaviors")
print(" 3. L(s,λ(n)) behaves differently from L(s,μ(n)) and L_DH despite all being")
print(" classified as 'anomalous'")
print(" 4. Statistical power may be limited by small sample sizes (n=2-3 per class)")
print()
print("This finding aligns with previous research (r84) that showed GEV shape")
print("parameters are not reliable classifiers for separating standard L-functions")
print("from those with analytic anomalies.")

print("\n" + "="*80)


COMPREHENSIVE ANALYSIS SUMMARY

1. HYPOTHESIS TESTING
--------------------------------------------------------------------------------
Hypothesis: The 'Anomalous-Resonant' class would have a GEV shape parameter
ξ that is statistically distinct from both 'Standard' and 'Control' classes.

Result: HYPOTHESIS NOT SUPPORTED

- One-way ANOVA: F = 0.4790, p = 0.6509 (not significant)
- No significant differences between any pair of classes (all p > 0.05)


2. KEY FINDINGS
--------------------------------------------------------------------------------

GEV Shape Parameter (ξ) by Class:
 Standard : Mean = -0.149, SD = 0.076
 Anomalous-Resonant : Mean = -0.057, SD = 0.117
 Control : Mean = -0.065, SD = 0.120


Individual Function Results:
 zeta (Standard ): ξ = -0.2030 [-0.2777, -0.1293]
 chi4 (Standard ): ξ = -0.0950 [-0.1764, -0.0017]
 liouville (Anomalous-Resonant ): ξ = 0.0622 [-0.0330, 0.1536]
 mobius (Anomalous-Resonant ): ξ = -0.0598 [-0.1586, 0.0291]
 L_DH (Anomalous-Resonant ): ξ = -0

In [20]:

# Additional diagnostic: Check if the issue is with R_comp definition
# Let's look at some sample R_comp time series

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

# Sample a subset of the t range for visualization
t_sample_start = 5000
t_sample_end = 6000
mask = (t_values >= t_sample_start) & (t_values < t_sample_end)
t_sample = t_values[mask]

for idx, func_name in enumerate(func_names):
 ax = axes[idx]
 R_comp_sample = R_comp_results[func_name][mask]
 class_name = class_names_map[func_name]
 color = color_map[class_name]
 
 ax.plot(t_sample, R_comp_sample, linewidth=0.5, color=color, alpha=0.7)
 ax.set_xlabel('t', fontsize=9)
 ax.set_ylabel('R_comp(t)', fontsize=9)
 ax.set_title(f'{func_name} ({class_name})', fontweight='bold', fontsize=10)
 ax.grid(True, alpha=0.3)
 
 # Add statistics
 mean_val = np.mean(R_comp_sample)
 max_val = np.max(R_comp_sample)
 ax.text(0.02, 0.98, f'Mean: {mean_val:.2f}\nMax: {max_val:.2f}', 
 transform=ax.transAxes,
 fontsize=8, verticalalignment='top',
 bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

# Hide extra subplots
for idx in range(len(func_names), 9):
 axes[idx].axis('off')

plt.suptitle(f'R_comp Time Series Sample (t ∈ [{t_sample_start}, {t_sample_end}])', 
 fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('R_comp_time_series_sample.png', dpi=300, bbox_inches='tight')
plt.close()

print("R_comp time series sample plot saved as 'R_comp_time_series_sample.png'")

# Print some diagnostic statistics
print("\nDiagnostic Statistics for Full R_comp Time Series:")
print("-"*80)
for func_name in func_names:
 R_comp = R_comp_results[func_name]
 print(f"{func_name:15s}: Mean={np.mean(R_comp):.4f}, Std={np.std(R_comp):.4f}, ", end='')
 print(f"Max={np.max(R_comp):.4f}, 99th%ile={np.percentile(R_comp, 99):.4f}")


R_comp time series sample plot saved as 'R_comp_time_series_sample.png'

Diagnostic Statistics for Full R_comp Time Series:
--------------------------------------------------------------------------------
zeta : Mean=1.0017, Std=1.1475, Max=9.7601, 99th%ile=5.6504
chi4 : Mean=1.0032, Std=1.1877, Max=10.9488, 99th%ile=6.0704
liouville : Mean=1.0016, Std=1.2177, Max=16.5256, 99th%ile=6.1935
mobius : Mean=1.0020, Std=0.9994, Max=11.6773, 99th%ile=4.9407
f_rand : Mean=1.0018, Std=1.2264, Max=18.3878, 99th%ile=6.2689
f_fully_rand : Mean=1.0001, Std=0.5211, Max=3.3631, 99th%ile=2.3991
L_DH : Mean=1.0024, Std=0.8770, Max=7.5650, 99th%ile=4.2033


In [21]:

# One more diagnostic: Compare the actual block maxima distributions more directly

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

class_names_ordered = ['Standard', 'Anomalous-Resonant', 'Control']

for idx, class_name in enumerate(class_names_ordered):
 ax = axes[idx]
 
 # Get all functions in this class
 funcs_in_class = [f for f in func_names if class_names_map[f] == class_name]
 
 # Plot distributions
 for func_name in funcs_in_class:
 block_max = block_maxima_dict[func_name]
 ax.hist(block_max, bins=25, alpha=0.5, label=func_name, density=True)
 
 ax.set_xlabel('Block Maximum R_comp', fontsize=11)
 ax.set_ylabel('Density', fontsize=11)
 ax.set_title(f'{class_name}', fontweight='bold', fontsize=12)
 ax.legend(fontsize=9)
 ax.grid(True, alpha=0.3)

plt.suptitle('Distribution of Block Maxima by Function Class', 
 fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('block_maxima_by_class.png', dpi=300, bbox_inches='tight')
plt.close()

print("Block maxima by class comparison plot saved as 'block_maxima_by_class.png'")

# Statistical summary of the distributions themselves (not just ξ)
print("\n" + "="*80)
print("DISTRIBUTIONAL CHARACTERISTICS OF BLOCK MAXIMA")
print("="*80)

for class_name in class_names_ordered:
 print(f"\n{class_name}:")
 print("-"*40)
 funcs_in_class = [f for f in func_names if class_names_map[f] == class_name]
 
 for func_name in funcs_in_class:
 block_max = block_maxima_dict[func_name]
 print(f" {func_name:15s}: ", end='')
 print(f"Range=[{np.min(block_max):.2f}, {np.max(block_max):.2f}], ", end='')
 print(f"IQR={np.percentile(block_max, 75) - np.percentile(block_max, 25):.2f}, ", end='')
 print(f"Skew={sp_stats.skew(block_max):.3f}")

print("\n" + "="*80)


Block maxima by class comparison plot saved as 'block_maxima_by_class.png'

DISTRIBUTIONAL CHARACTERISTICS OF BLOCK MAXIMA

Standard:
----------------------------------------
 zeta : Range=[4.71, 9.76], IQR=1.35, Skew=0.260
 chi4 : Range=[5.26, 10.95], IQR=1.55, Skew=0.569

Anomalous-Resonant:
----------------------------------------
 liouville : Range=[4.86, 16.53], IQR=3.36, Skew=0.877
 mobius : Range=[3.76, 11.68], IQR=1.76, Skew=0.768
 L_DH : Range=[3.45, 7.56], IQR=1.04, Skew=0.290

Control:
----------------------------------------
 f_rand : Range=[4.73, 18.39], IQR=3.50, Skew=0.843
 f_fully_rand : Range=[2.09, 3.36], IQR=0.37, Skew=0.419



In [22]:

# Create the final figure as specified: a single figure with key results
# This should be the definitive visualization for the analysis

fig = plt.figure(figsize=(10, 10))

# Single panel showing ξ estimates with CIs
ax = fig.add_subplot(111)

# Organize by class
positions = []
current_pos = 0

for class_name in ['Standard', 'Anomalous-Resonant', 'Control']:
 funcs_in_class = [f for f in func_names if class_names_map[f] == class_name]
 
 for func_name in funcs_in_class:
 positions.append(current_pos)
 current_pos += 1
 
 current_pos += 0.5 # Add gap between classes

# Plot each function
plot_idx = 0
for class_name in ['Standard', 'Anomalous-Resonant', 'Control']:
 funcs_in_class = [f for f in func_names if class_names_map[f] == class_name]
 color = color_map[class_name]
 
 for func_name in funcs_in_class:
 result = gev_results[func_name]
 xi = result['xi']
 xi_ci = result['xi_ci']
 
 pos = positions[plot_idx]
 
 # Plot point and CI
 ax.errorbar(pos, xi, 
 yerr=[[xi - xi_ci[0]], [xi_ci[1] - xi]], 
 fmt='o', markersize=12, capsize=10, capthick=2.5,
 color=color, ecolor=color, linewidth=2.5, alpha=0.9,
 zorder=3)
 
 plot_idx += 1

# Reference line at ξ=0
ax.axhline(y=0, color='black', linestyle='--', linewidth=1.5, alpha=0.6, 
 label='ξ = 0 (Gumbel)', zorder=1)

# Styling
ax.set_xticks(positions)
labels = []
for class_name in ['Standard', 'Anomalous-Resonant', 'Control']:
 funcs_in_class = [f for f in func_names if class_names_map[f] == class_name]
 labels.extend(funcs_in_class)
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=11)
ax.set_ylabel('GEV Shape Parameter (ξ)', fontsize=13, fontweight='bold')
ax.set_title('Extreme Value Analysis: GEV Shape Parameter by Function Class\n' + 
 f'ANOVA: F={F_stat:.3f}, p={p_value:.3f} (not significant)', 
 fontsize=13, fontweight='bold', pad=15)
ax.grid(True, alpha=0.3, axis='y', zorder=0)

# Add class separators and labels
sep_positions = [1.5, 4.5]
for sep_pos in sep_positions:
 ax.axvline(x=sep_pos, color='black', linestyle='-', linewidth=2, alpha=0.4, zorder=2)

# Class labels at top
y_label_pos = ax.get_ylim()[1] * 0.93

class_x_positions = {
 'Standard': 0.5,
 'Anomalous-Resonant': 3,
 'Control': 5.5
}

for class_name, x_pos in class_x_positions.items():
 ax.text(x_pos, y_label_pos, class_name, 
 ha='center', va='bottom', fontsize=11, fontweight='bold',
 color=color_map[class_name],
 bbox=dict(boxstyle='round,pad=0.5', facecolor='white', 
 edgecolor=color_map[class_name], linewidth=2, alpha=0.9),
 zorder=4)

# Add summary statistics box
summary_text = "Class Mean ξ (±SD):\n"
for class_name in ['Standard', 'Anomalous-Resonant', 'Control']:
 xi_vals = xi_by_class[class_name]
 mean_xi = np.mean(xi_vals)
 std_xi = np.std(xi_vals, ddof=1)
 summary_text += f"{class_name}: {mean_xi:.3f} ± {std_xi:.3f}\n"

ax.text(0.02, 0.02, summary_text.strip(),
 transform=ax.transAxes,
 fontsize=9, verticalalignment='bottom', horizontalalignment='left',
 bbox=dict(boxstyle='round', facecolor='white', alpha=0.95, 
 edgecolor='black', linewidth=1.5),
 family='monospace', zorder=5)

plt.tight_layout()
plt.savefig('final_figure.png', dpi=300, bbox_inches='tight')
plt.close()

print("="*80)
print("FINAL FIGURE CREATED")
print("="*80)
print("Saved as: final_figure.png")
print()
print("This figure shows:")
print(" - GEV shape parameter (ξ) estimates with 95% confidence intervals")
print(" - Functions grouped by class (Standard, Anomalous-Resonant, Control)")
print(" - Reference line at ξ=0 (Gumbel distribution)")
print(" - ANOVA results showing no significant difference between classes")
print("="*80)


FINAL FIGURE CREATED
Saved as: final_figure.png

This figure shows:
 - GEV shape parameter (ξ) estimates with 95% confidence intervals
 - Functions grouped by class (Standard, Anomalous-Resonant, Control)
 - Reference line at ξ=0 (Gumbel distribution)
 - ANOVA results showing no significant difference between classes


In [23]:

# Final comprehensive documentation

final_summary = """
================================================================================
COMPREHENSIVE ANALYSIS REPORT
================================================================================

RESEARCH QUESTION:
Do the extreme value distributions of R_comp metrics show clear separation
between three function classes (Standard, Anomalous-Resonant, Control)?

HYPOTHESIS:
The "Anomalous-Resonant" class will have a GEV shape parameter ξ that is 
statistically distinct from both "Standard" and "Control" classes.

================================================================================
METHODS
================================================================================

1. FUNCTION CLASSES DEFINED:
 - Standard: ζ(s), L(s,χ₄) (real character mod 5)
 - Anomalous-Resonant: L(s,λ(n)), L(s,μ(n)), validated historical L_DH
 - Control: f_rand (seed=42), f_fully_rand (seed=42)

2. COMPUTATIONAL PARAMETERS:
 - Truncation length: N = 10^5
 - Time range: t ∈ [1000, 20000]
 - Sampling resolution: dt = 2π/log(N) ≈ 0.546
 - Number of time points: 34,815
 - R_comp window size: 100 points
 
3. EXTREME VALUE ANALYSIS:
 - Block maxima extraction: 200 blocks per function
 - GEV fitting: Maximum likelihood estimation
 - Confidence intervals: Bootstrap (1000 samples, 95% CI)
 
4. STATISTICAL TESTS:
 - One-way ANOVA for class differences
 - Post-hoc pairwise t-tests (Welch's)
 - Cohen's d effect sizes

================================================================================
RESULTS
================================================================================

1. GEV SHAPE PARAMETER ESTIMATES:

Function Class ξ 95% CI
-------- ----- -- ------
zeta Standard -0.203 [-0.278, -0.129]
chi4 Standard -0.095 [-0.176, -0.002]
liouville Anomalous-Resonant 0.062 [-0.033, 0.154]
mobius Anomalous-Resonant -0.060 [-0.159, 0.029]
L_DH Anomalous-Resonant -0.172 [-0.330, -0.097]
f_rand Control 0.020 [-0.097, 0.128]
f_fully_rand Control -0.149 [-0.241, -0.044]

2. CLASS-LEVEL STATISTICS:

Class Mean ξ SD Range
----- ------ -- -----
Standard -0.149 0.076 [-0.203, -0.095]
Anomalous-Resonant -0.057 0.117 [-0.172, 0.062]
Control -0.065 0.120 [-0.149, 0.020]

3. STATISTICAL TESTS:

One-way ANOVA:
 F-statistic = 0.479
 p-value = 0.651
 Conclusion: NO SIGNIFICANT DIFFERENCE between classes

Post-hoc Pairwise Comparisons (Welch's t-test):
 Standard vs Anomalous-Resonant: p = 0.366 (not significant)
 Standard vs Control: p = 0.502 (not significant)
 Anomalous-Resonant vs Control: p = 0.948 (not significant)

4. EFFECT SIZES (Cohen's d):
 Standard vs Anomalous-Resonant: d = -0.875 (large)
 Standard vs Control: d = -0.840 (large)
 Anomalous-Resonant vs Control: d = 0.067 (negligible)

Note: Large effect sizes but non-significant p-values indicate high
within-group variability and/or small sample sizes.

5. CONFIDENCE INTERVAL OVERLAPS:
 - 17 out of 21 function pairs show overlapping CIs
 - Only 4 pairs are clearly distinct:
 * zeta vs liouville
 * zeta vs f_rand
 * liouville vs L_DH
 * liouville vs f_fully_rand

================================================================================
CONCLUSION
================================================================================

HYPOTHESIS: NOT SUPPORTED

The extreme value distributions of R_comp metrics do NOT show clear statistical
separation between the three function classes. Specifically:

1. No significant differences in mean ξ between classes (ANOVA p = 0.651)

2. High within-class heterogeneity, especially in Anomalous-Resonant class:
 - L(s,λ(n)) shows positive ξ (heavier tail)
 - L(s,μ(n)) shows near-zero ξ
 - L_DH shows negative ξ (lighter tail)

3. Confidence intervals overlap extensively between classes

4. The Anomalous-Resonant class does not exhibit a unique tail behavior

This finding is consistent with prior research (r84) demonstrating that 
GEV shape parameters of resonance metrics are not reliable classifiers
for distinguishing standard L-functions from those with analytic anomalies.

================================================================================
KEY INSIGHTS
================================================================================

1. HETEROGENEITY WITHIN "ANOMALOUS" CLASS:
 Functions with known analytic anomalies (off-line zeros or poles) do not
 form a homogeneous group in terms of extreme value statistics. This suggests
 that the mechanism of resonance differs between these functions.

2. L(s,λ(n)) OUTLIER BEHAVIOR:
 The Liouville function L(s,λ(n)) consistently shows distinct behavior from
 other anomalous functions, with positive ξ indicating heavier-tailed extremes.
 This aligns with r83 findings of high resonance via phase-alignment.

3. CONTROL FUNCTIONS SHOW VARIABILITY:
 Even purely random functions (f_rand, f_fully_rand) show different extreme
 value behaviors, indicating that the R_comp metric is sensitive to the
 structure of coefficients beyond multiplicativity.

4. STATISTICAL POWER LIMITATIONS:
 Small sample sizes (n=2-3 per class) limit the power to detect differences,
 even when effect sizes are large.

================================================================================
IMPLICATIONS
================================================================================

1. The R_comp metric's extreme value distribution (via GEV ξ) is NOT a 
 reliable classifier for separating standard L-functions from those with
 analytic anomalies.

2. Alternative approaches needed:
 - Multi-scale diagnostics (as suggested in research program)
 - Phase decomposition analysis
 - Persistence measures across scales
 - Direct detection of off-line zero signatures

3. The three-class framework may be too coarse. Functions within the
 "Anomalous-Resonant" class exhibit fundamentally different behaviors that
 should not be grouped together.

================================================================================
"""

print(final_summary)

# Save to file
with open('final_analysis_report.txt', 'w') as f:
 f.write(final_summary)

print("\nFull analysis report saved to 'final_analysis_report.txt'")



COMPREHENSIVE ANALYSIS REPORT

RESEARCH QUESTION:
Do the extreme value distributions of R_comp metrics show clear separation
between three function classes (Standard, Anomalous-Resonant, Control)?

HYPOTHESIS:
The "Anomalous-Resonant" class will have a GEV shape parameter ξ that is 
statistically distinct from both "Standard" and "Control" classes.

METHODS

1. FUNCTION CLASSES DEFINED:
 - Standard: ζ(s), L(s,χ₄) (real character mod 5)
 - Anomalous-Resonant: L(s,λ(n)), L(s,μ(n)), validated historical L_DH
 - Control: f_rand (seed=42), f_fully_rand (seed=42)

2. COMPUTATIONAL PARAMETERS:
 - Truncation length: N = 10^5
 - Time range: t ∈ [1000, 20000]
 - Sampling resolution: dt = 2π/log(N) ≈ 0.546
 - Number of time points: 34,815
 - R_comp window size: 100 points

3. EXTREME VALUE ANALYSIS:
 - Block maxima extraction: 200 blocks per function
 - GEV fitting: Maximum likelihood estimation
 - Confidence intervals: Bootstrap (1000 samples, 95% CI)

4. STATISTICAL TESTS:
 - One-way ANOVA for